# Tech Challenge Fase 2
## 03.1 — Gold Alunos

### Objetivo

Transformar a base Silver de alunos em indicadores agregados por município e UF.

A Gold não mantém o nível individual do aluno para consumo analítico.  
Ela cria medidas como:

- quantidade de alunos;
- quantidade de presentes;
- taxa de participação;
- quantidade e percentual de alfabetizados;
- proficiência média;
- distribuição por dependência administrativa;
- quantidade de escolas.

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Leitura da gold_metadata

In [0]:
metadata = pd.read_parquet(CONFIG_PATH / "gold_metadata")

metadata_alunos = (
    metadata[
        metadata["produto"] == "gold_alunos"
    ]
    .sort_values("ano")
    .reset_index(drop=True)
)

display(metadata_alunos)

## 5. Leitura das bases Silver de alunos

In [0]:
bases = []

for _, row in metadata_alunos.iterrows():
    caminho = Path(row["silver_path"]) / row["silver_file_name"]
    df = ler_csv(caminho)
    df["ANO"] = int(row["ano"])
    bases.append(df)

df_alunos = pd.concat(bases, ignore_index=True)

print("Base de alunos:", df_alunos.shape)

## 6. Padronização e criação de indicadores

In [0]:
for coluna in ["CO_UF", "CO_MUNICIPIO", "ID_ESCOLA"]:
    if coluna in df_alunos.columns:
        df_alunos[coluna] = normalizar_codigo(df_alunos[coluna])

for coluna in [
    "IN_PRESENCA_LP",
    "IN_PREENCHIMENTO_LP",
    "VL_PROFICIENCIA_LP",
    "IN_ALFABETIZADO"
]:
    if coluna in df_alunos.columns:
        df_alunos[coluna] = converter_numero(df_alunos[coluna])

df_alunos["presente_lp"] = np.where(
    df_alunos["IN_PRESENCA_LP"] == 1,
    1,
    0
)

df_alunos["alfabetizado"] = np.where(
    df_alunos["IN_ALFABETIZADO"] == 1,
    1,
    0
)

## 7. Agregação por município

In [0]:
df_gold_alunos_municipio = (
    df_alunos
    .groupby(
        ["ANO", "CO_UF", "SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO"],
        dropna=False
    )
    .agg(
        qtd_alunos=("ID_ALUNO", "nunique"),
        qtd_escolas=("ID_ESCOLA", "nunique"),
        qtd_presentes_lp=("presente_lp", "sum"),
        qtd_alfabetizados=("alfabetizado", "sum"),
        proficiencia_media_lp=("VL_PROFICIENCIA_LP", "mean")
    )
    .reset_index()
)

df_gold_alunos_municipio["taxa_participacao_lp"] = (
    df_gold_alunos_municipio["qtd_presentes_lp"]
    / df_gold_alunos_municipio["qtd_alunos"]
    * 100
)

df_gold_alunos_municipio["taxa_alfabetizacao_alunos"] = (
    df_gold_alunos_municipio["qtd_alfabetizados"]
    / df_gold_alunos_municipio["qtd_presentes_lp"]
    * 100
)

df_gold_alunos_municipio["_gold_processed_at"] = datetime.now().isoformat()

if df_gold_alunos_municipio.empty:
    print("Nenhum indicador municipal de alunos foi gerado.")
else:
    display(df_gold_alunos_municipio.head())

## 8. Agregação por UF

In [0]:
df_gold_alunos_uf = (
    df_alunos
    .groupby(
        ["ANO", "CO_UF", "SG_UF"],
        dropna=False
    )
    .agg(
        qtd_alunos=("ID_ALUNO", "nunique"),
        qtd_escolas=("ID_ESCOLA", "nunique"),
        qtd_presentes_lp=("presente_lp", "sum"),
        qtd_alfabetizados=("alfabetizado", "sum"),
        proficiencia_media_lp=("VL_PROFICIENCIA_LP", "mean")
    )
    .reset_index()
)

df_gold_alunos_uf["taxa_participacao_lp"] = (
    df_gold_alunos_uf["qtd_presentes_lp"]
    / df_gold_alunos_uf["qtd_alunos"]
    * 100
)

df_gold_alunos_uf["taxa_alfabetizacao_alunos"] = (
    df_gold_alunos_uf["qtd_alfabetizados"]
    / df_gold_alunos_uf["qtd_presentes_lp"]
    * 100
)

df_gold_alunos_uf["_gold_processed_at"] = datetime.now().isoformat()

## 9. Persistência

In [0]:
for ano in [2023, 2024, 2025]:
    registro = metadata_alunos[metadata_alunos["ano"] == ano].iloc[0]
    destino = Path(registro["gold_output_path"])

    salvar_csv(
        df_gold_alunos_municipio[
            df_gold_alunos_municipio["ANO"] == ano
        ],
        destino,
        registro["gold_file_name"]
    )

salvar_csv(
    df_gold_alunos_uf,
    GOLD_PATH / "alunos_ufs",
    "GOLD_ALUNOS_UFS.csv"
)

print("Gold Alunos salva com sucesso.")